# Getting started with ISA-PHM

Explore an ISA-PHM investigation using your own dataset. This introduction covers metadata only; it does not load measurement CSV files. Install the Wrapper in your notebook environment first (see `notebooks/README.md`).


## 1. Open your dataset

Set `isa_json` to the ISA-PHM JSON file at the root of your dataset. Its parent directory is used to resolve dataset files and attachments.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from isa_phm import ISAWrapper

isa_json = Path('path/to/your-dataset.isa-phm.json').expanduser()
if not isa_json.is_file():
    raise FileNotFoundError(f'Set isa_json to your dataset JSON file: {isa_json}')

wrapper = ISAWrapper(isa_json, data_root=isa_json.parent, strict_validation=False)


## 2. Investigation

Inspect the investigation's title, description, experiment type and study count.


In [ ]:
display(wrapper.summary())


## 3. Contacts and publications

These tables can be empty if the information is not recorded.


In [ ]:
display(wrapper.contacts())
display(wrapper.publications())


## 4. Studies

A dataset may have one or more studies. The example below selects the first study for the remaining exploration.


In [ ]:
studies = wrapper.list_studies()
display(pd.DataFrame([item.model_dump() for item in studies]))
if not studies:
    raise ValueError('This investigation has no studies to explore.')
study = wrapper.study(studies[0].title)
print('Selected study:', study.title)


## 5. Study and sensors

Inspect the selected study's metadata and its sensor catalogue. Available information depends on what was recorded in your dataset.


In [ ]:
display(study.overview().model_dump())
display(study.sensor_catalog())


## 6. Test setup images and datasheets

List the attachments referenced by the selected study, including unavailable ones. If an image is available, preview the first one. This is an attachment and sensor overview, not a complete structured test-setup model.


In [ ]:
attachments = wrapper.attachments(study=study.title)
display(pd.DataFrame([
    {'kind': item.kind, 'owner': item.owner, 'status': item.status,
     'path': str(item.path) if item.path else item.reference}
    for item in attachments
], columns=['kind', 'owner', 'status', 'path']))

image = next((item for item in attachments if item.kind == 'image' and item.status == 'available'), None)
if image is not None:
    image.display()
else:
    print('No available setup image is recorded for this study.')


## Next steps

For measurement data and analysis, choose the [diagnostic workflow](../03_workflows/02_diagnostic_workflow.ipynb) or the [prognostic workflow](../03_workflows/03_prognostic_workflow.ipynb). These examples have dataset-specific assumptions; see `notebooks/README.md`.
